In [0]:
weather_path = "/Volumes/workspace/default/farewise_data/weather/*.csv"

weather_df = (
    spark.read
    .option("header", "true")
    .option("skipRows", 3)
    .option("inferSchema", "true")
    .csv(weather_path)
)

weather_df.printSchema()
display(weather_df.limit(10))
print("Weather records:", weather_df.count())

root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m (°C): double (nullable = true)
 |-- precipitation (mm): double (nullable = true)
 |-- rain (mm): double (nullable = true)
 |-- snowfall (cm): double (nullable = true)
 |-- wind_speed_10m (km/h): double (nullable = true)



time,temperature_2m (°C),precipitation (mm),rain (mm),snowfall (cm),wind_speed_10m (km/h)
2009-01-01T00:00:00.000Z,-5.5,0.0,0.0,0.0,32.8
2009-01-01T01:00:00.000Z,-5.5,0.0,0.0,0.0,34.1
2009-01-01T02:00:00.000Z,-5.6,0.0,0.0,0.0,33.6
2009-01-01T03:00:00.000Z,-6.2,0.0,0.0,0.0,33.9
2009-01-01T04:00:00.000Z,-6.2,0.0,0.0,0.0,31.3
2009-01-01T05:00:00.000Z,-6.3,0.0,0.0,0.0,30.8
2009-01-01T06:00:00.000Z,-6.4,0.0,0.0,0.0,30.3
2009-01-01T07:00:00.000Z,-6.4,0.0,0.0,0.0,29.8
2009-01-01T08:00:00.000Z,-6.7,0.0,0.0,0.0,29.0
2009-01-01T09:00:00.000Z,-7.3,0.0,0.0,0.0,27.8


Weather records: 56928


In [0]:
from pyspark.sql import functions as F

weather_df.select(
    F.count("*").alias("total_records"),
    F.min("time").alias("earliest_date"),
    F.max("time").alias("latest_date"),
    F.countDistinct("time").alias("unique_hours")
).show(truncate=False)

+-------------+-------------------+-------------------+------------+
|total_records|earliest_date      |latest_date        |unique_hours|
+-------------+-------------------+-------------------+------------+
|56928        |2009-01-01 00:00:00|2015-06-30 23:00:00|56928       |
+-------------+-------------------+-------------------+------------+



In [0]:
from pyspark.sql import functions as F

weather_df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in weather_df.columns
]).show(truncate=False)

+----+-------------------+------------------+---------+-------------+---------------------+
|time|temperature_2m (°C)|precipitation (mm)|rain (mm)|snowfall (cm)|wind_speed_10m (km/h)|
+----+-------------------+------------------+---------+-------------+---------------------+
|0   |0                  |0                 |0        |0            |0                    |
+----+-------------------+------------------+---------+-------------+---------------------+



In [0]:
dbutils.fs.ls("/Volumes/workspace/default/farewise_data/taxi/")

taxi_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/farewise_data/taxi/train.csv")
)

print("Total records:", taxi_df.count())

Total records: 55423856


In [0]:
from pyspark.sql import functions as F

taxi_df.select([
    F.count(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1
        )
    ).alias(c)
    for c in taxi_df.columns
]).show(truncate=False)

+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|key|fare_amount|pickup_datetime|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|passenger_count|
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+
|0  |0          |0              |0               |0              |376              |376             |0              |
+---+-----------+---------------+----------------+---------------+-----------------+----------------+---------------+



In [0]:
taxi_typed = (
    taxi_df
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("passenger_count", F.col("passenger_count").cast("int"))
    .withColumn("pickup_latitude", F.col("pickup_latitude").cast("double"))
    .withColumn("pickup_longitude", F.col("pickup_longitude").cast("double"))
    .withColumn("dropoff_latitude", F.col("dropoff_latitude").cast("double"))
    .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double"))
)

In [0]:
taxi_typed.select(
    F.count(F.when(F.col("fare_amount") <= 0, 1)).alias("invalid_fares"),
    F.count(F.when(
        (F.col("passenger_count") <= 0) |
        (F.col("passenger_count") > 6), 1
    )).alias("invalid_passengers"),
    F.count(F.when(
        F.col("pickup_latitude").isNull() |
        (~F.col("pickup_latitude").between(-90, 90)) |
        (~F.col("pickup_longitude").between(-180, 180)) |
        (~F.col("dropoff_latitude").between(-90, 90)) |
        (~F.col("dropoff_longitude").between(-180, 180)),
        1
    )).alias("invalid_coordinates")
).show()

+-------------+------------------+-------------------+
|invalid_fares|invalid_passengers|invalid_coordinates|
+-------------+------------------+-------------------+
|         3834|            195532|               2745|
+-------------+------------------+-------------------+



In [0]:
taxi_typed.select(
    F.min("pickup_datetime").alias("earliest_pickup"),
    F.max("pickup_datetime").alias("latest_pickup")
).show(truncate=False)

+-----------------------+-----------------------+
|earliest_pickup        |latest_pickup          |
+-----------------------+-----------------------+
|2009-01-01 00:00:27 UTC|2015-06-30 23:59:54 UTC|
+-----------------------+-----------------------+



In [0]:

from pyspark.sql import functions as F

taxi_clean = taxi_typed.filter(
    (F.col("fare_amount") > 0) &
    F.col("passenger_count").between(1, 6) &
    F.col("pickup_latitude").between(40, 41) &
    F.col("pickup_longitude").between(-75, -73) &
    F.col("dropoff_latitude").between(40, 41) &
    F.col("dropoff_longitude").between(-75, -73)
)

print("Original records:", taxi_typed.count())
print("Cleaned records:", taxi_clean.count())
print("Removed records:",
      taxi_typed.count() - taxi_clean.count())


Original records: 55423856
Cleaned records: 54033721
Removed records: 1390135


In [0]:

# Use UTC consistently for the weather join
spark.conf.set("spark.sql.session.timeZone", "UTC")

taxi_clean = (
    taxi_clean
    .withColumn(
        "pickup_ts",
        F.to_timestamp(
            F.regexp_replace(
                F.col("pickup_datetime"),
                " UTC$", ""
            ),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        "weather_hour",
        F.date_trunc("hour", F.col("pickup_ts"))
    )
)

taxi_clean.select(
    "pickup_datetime",
    "pickup_ts",
    "weather_hour"
).show(5, truncate=False)


+-----------------------+-------------------+-------------------+
|pickup_datetime        |pickup_ts          |weather_hour       |
+-----------------------+-------------------+-------------------+
|2009-06-15 17:26:21 UTC|2009-06-15 17:26:21|2009-06-15 17:00:00|
|2010-01-05 16:52:16 UTC|2010-01-05 16:52:16|2010-01-05 16:00:00|
|2011-08-18 00:35:00 UTC|2011-08-18 00:35:00|2011-08-18 00:00:00|
|2012-04-21 04:30:42 UTC|2012-04-21 04:30:42|2012-04-21 04:00:00|
|2010-03-09 07:51:00 UTC|2010-03-09 07:51:00|2010-03-09 07:00:00|
+-----------------------+-------------------+-------------------+
only showing top 5 rows


In [0]:

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Reload weather data
weather_df = (
    spark.read
    .option("header", True)
    .option("skipRows", 3)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/farewise_data/weather/*.csv")
)

# Prepare hourly weather data
weather_ready = weather_df.withColumnRenamed(
    "time", "weather_hour"
)

# Join taxi records with hourly weather
combined_df = taxi_clean.join(
    F.broadcast(weather_ready),
    on="weather_hour",
    how="left"
)

# Preview combined data
display(
    combined_df.select(
        "pickup_datetime",
        "fare_amount",
        "temperature_2m (°C)",
        "precipitation (mm)"
    ).limit(10)
)


pickup_datetime,fare_amount,temperature_2m (°C),precipitation (mm)
2009-06-15 17:26:21 UTC,4.5,19.7,0.0
2010-01-05 16:52:16 UTC,16.9,-2.7,0.0
2011-08-18 00:35:00 UTC,5.7,26.3,0.0
2012-04-21 04:30:42 UTC,7.7,15.2,0.0
2010-03-09 07:51:00 UTC,5.3,7.6,0.0
2011-01-06 09:50:45 UTC,12.1,-5.2,0.0
2012-11-20 20:35:00 UTC,7.5,9.2,0.0
2012-01-04 17:22:00 UTC,16.5,-5.4,0.0
2012-12-03 13:10:00 UTC,9.0,8.0,0.0
2009-09-02 01:11:00 UTC,8.9,19.3,0.0


In [0]:

combined_df.select(
    F.count("*").alias("total_joined"),
    F.count("temperature_2m (°C)").alias("matched_weather"),
    F.sum(
        F.when(
            F.col("temperature_2m (°C)").isNull(), 1
        ).otherwise(0)
    ).alias("unmatched_weather")
).show()


+------------+---------------+-----------------+
|total_joined|matched_weather|unmatched_weather|
+------------+---------------+-----------------+
|    54033721|       54033721|                0|
+------------+---------------+-----------------+



In [0]:

# Save all cleaned and joined records
combined_df.write \
    .mode("overwrite") \
    .parquet(
        "/Volumes/workspace/default/farewise_data/processed/"
    )

print("Combined dataset saved successfully!")


Combined dataset saved successfully!


In [0]:
combined_df.printSchema()

root
 |-- weather_hour: timestamp (nullable = true)
 |-- key: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- temperature_2m (°C): double (nullable = true)
 |-- precipitation (mm): double (nullable = true)
 |-- rain (mm): double (nullable = true)
 |-- snowfall (cm): double (nullable = true)
 |-- wind_speed_10m (km/h): double (nullable = true)



In [0]:
display(combined_df)

weather_hour,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_ts,temperature_2m (°C),precipitation (mm),rain (mm),snowfall (cm),wind_speed_10m (km/h)
2009-06-15T17:00:00.000Z,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21 UTC,-73.844311,40.721319,-73.84161,40.712278,1,2009-06-15T17:26:21.000Z,19.7,0.0,0.0,0.0,7.4
2010-01-05T16:00:00.000Z,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16 UTC,-74.016048,40.711303,-73.979268,40.782004,1,2010-01-05T16:52:16.000Z,-2.7,0.0,0.0,0.0,17.8
2011-08-18T00:00:00.000Z,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00 UTC,-73.982738,40.76127,-73.991242,40.750562,2,2011-08-18T00:35:00.000Z,26.3,0.0,0.0,0.0,8.0
2012-04-21T04:00:00.000Z,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42 UTC,-73.98713,40.733143,-73.991567,40.758092,1,2012-04-21T04:30:42.000Z,15.2,0.0,0.0,0.0,5.5
2010-03-09T07:00:00.000Z,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00 UTC,-73.968095,40.768008,-73.956655,40.783762,1,2010-03-09T07:51:00.000Z,7.6,0.0,0.0,0.0,11.9
2011-01-06T09:00:00.000Z,2011-01-06 09:50:45.0000002,12.1,2011-01-06 09:50:45 UTC,-74.000964,40.73163,-73.972892,40.758233,1,2011-01-06T09:50:45.000Z,-5.2,0.0,0.0,0.0,3.9
2012-11-20T20:00:00.000Z,2012-11-20 20:35:00.0000001,7.5,2012-11-20 20:35:00 UTC,-73.980002,40.751662,-73.973802,40.764842,1,2012-11-20T20:35:00.000Z,9.2,0.0,0.0,0.0,4.6
2012-01-04T17:00:00.000Z,2012-01-04 17:22:00.00000081,16.5,2012-01-04 17:22:00 UTC,-73.9513,40.774138,-73.990095,40.751048,1,2012-01-04T17:22:00.000Z,-5.4,0.0,0.0,0.0,10.1
2012-12-03T13:00:00.000Z,2012-12-03 13:10:00.000000125,9.0,2012-12-03 13:10:00 UTC,-74.006462,40.726713,-73.993078,40.731628,1,2012-12-03T13:10:00.000Z,8.0,0.0,0.0,0.0,8.0
2009-09-02T01:00:00.000Z,2009-09-02 01:11:00.00000083,8.9,2009-09-02 01:11:00 UTC,-73.980658,40.733873,-73.99154,40.758138,2,2009-09-02T01:11:00.000Z,19.3,0.0,0.0,0.0,5.4


In [0]:
print("Total rows:", combined_df.count())
print("Total columns:", len(combined_df.columns))

Total rows: 54033721
Total columns: 15


In [0]:
# Load  saved combined dataset
saved_df = spark.read.parquet(
    "/Volumes/workspace/default/farewise_data/processed/"
)

# Show summary statistics
display(
    saved_df.select(
        "fare_amount",
        "passenger_count",
        "temperature_2m (°C)",
        "precipitation (mm)"
    ).summary()
)

summary,fare_amount,passenger_count,temperature_2m (°C),precipitation (mm)
count,54033721,54033721,54033721,54033721
mean,11.324100118509833,1.6912732698901118,12.430037166976495,0.1229654774284795
stddev,20.835544607770082,1.3071210709268795,10.150191489756551,0.5065848892378189
min,0.01,1,-19.1,0.0
25%,6.0,1,4.4,0.0
50%,8.5,1,12.8,0.0
75%,12.5,2,20.9,0.0
max,93963.36,6,39.2,15.9


In [0]:
print("Total records:", saved_df.count())
print("Total columns:", len(saved_df.columns))

saved_df.printSchema()

display(saved_df.limit(100))

Total records: 54033721
Total columns: 15
root
 |-- weather_hour: timestamp (nullable = true)
 |-- key: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- temperature_2m (°C): double (nullable = true)
 |-- precipitation (mm): double (nullable = true)
 |-- rain (mm): double (nullable = true)
 |-- snowfall (cm): double (nullable = true)
 |-- wind_speed_10m (km/h): double (nullable = true)



weather_hour,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_ts,temperature_2m (°C),precipitation (mm),rain (mm),snowfall (cm),wind_speed_10m (km/h)
2009-07-01T23:00:00.000Z,2009-07-01 23:53:00.000000110,26.9,2009-07-01 23:53:00 UTC,-74.000163,40.738793,-73.927622,40.866605,1,2009-07-01T23:53:00.000Z,23.4,0.6,0.6,0.0,9.1
2013-03-13T19:00:00.000Z,2013-03-13 19:45:48.0000003,14.0,2013-03-13 19:45:48 UTC,-73.996235,40.723556,-73.987624,40.768753,2,2013-03-13T19:45:48.000Z,8.6,0.0,0.0,0.0,18.4
2014-11-30T15:00:00.000Z,2014-11-30 15:23:00.00000040,12.5,2014-11-30 15:23:00 UTC,-74.00044,40.757837,-73.978892,40.752677,1,2014-11-30T15:23:00.000Z,8.2,0.0,0.0,0.0,8.9
2011-06-26T21:00:00.000Z,2011-06-26 21:39:00.00000085,17.7,2011-06-26 21:39:00 UTC,-73.925322,40.762082,-73.999578,40.728563,5,2011-06-26T21:39:00.000Z,25.7,0.0,0.0,0.0,9.5
2013-05-10T23:00:00.000Z,2013-05-10 23:06:00.00000068,52.0,2013-05-10 23:06:00 UTC,-73.783782,40.6487,-74.006915,40.734832,1,2013-05-10T23:06:00.000Z,20.8,0.0,0.0,0.0,11.6
2010-09-16T19:00:00.000Z,2010-09-16 19:14:00.000000196,10.5,2010-09-16 19:14:00 UTC,-74.000462,40.729947,-73.991407,40.749623,1,2010-09-16T19:14:00.000Z,21.1,0.3,0.3,0.0,18.1
2011-03-28T14:00:00.000Z,2011-03-28 14:19:00.000000161,9.7,2011-03-28 14:19:00 UTC,-73.971838,40.7973,-73.988267,40.764458,1,2011-03-28T14:19:00.000Z,-0.3,0.0,0.0,0.0,18.8
2012-03-16T19:00:00.000Z,2012-03-16 19:06:01.0000004,10.1,2012-03-16 19:06:01 UTC,-73.964161,40.792605,-73.979411,40.763924,2,2012-03-16T19:06:01.000Z,8.1,0.0,0.0,0.0,2.5
2015-04-26T22:00:00.000Z,2015-04-26 22:30:57.0000007,16.5,2015-04-26 22:30:57 UTC,-73.98241424560547,40.74559020996094,-73.96533966064453,40.79067611694336,1,2015-04-26T22:30:57.000Z,13.8,0.0,0.0,0.0,4.7
2009-12-13T13:00:00.000Z,2009-12-13 13:45:25.0000003,37.47,2009-12-13 13:45:25 UTC,-73.981977,40.767573,-73.870868,40.773987,2,2009-12-13T13:45:25.000Z,-0.9,0.0,0.0,0.0,6.9


In [0]:

from pyspark.sql import functions as F

saved_df.select(
    F.count(
        F.when(F.col("fare_amount") > 500, 1)
    ).alias("fares_above_500"),

    F.count(
        F.when(F.col("fare_amount") < 2.50, 1)
    ).alias("fares_below_2_50"),

    F.count(
        F.when(F.col("fare_amount") > 20, 1)
    ).alias("fares_above_20"),

    F.count(
        F.when(F.col("fare_amount") <= 20, 1)
    ).alias("fares_20_or_below")
).show()


+---------------+----------------+--------------+-----------------+
|fares_above_500|fares_below_2_50|fares_above_20|fares_20_or_below|
+---------------+----------------+--------------+-----------------+
|             21|             709|       5519749|         48513972|
+---------------+----------------+--------------+-----------------+



In [0]:
display(
    saved_df
    .filter(F.col("fare_amount") > 500)
    .select(
        "fare_amount",
        "pickup_datetime",
        "pickup_latitude",
        "pickup_longitude",
        "dropoff_latitude",
        "dropoff_longitude",
        "passenger_count"
    )
    .orderBy(F.col("fare_amount").desc())
)

fare_amount,pickup_datetime,pickup_latitude,pickup_longitude,dropoff_latitude,dropoff_longitude,passenger_count
93963.36,2015-02-24 08:57:37 UTC,40.71235275268555,-73.99446105957031,40.702903747558594,-74.01254272460938,2
75747.02,2015-05-03 09:35:20 UTC,40.792354583740234,-73.96794891357422,40.738975524902344,-73.99089813232422,1
61550.86,2013-08-20 11:52:52 UTC,40.755965,-73.967711,40.747666,-73.945572,1
9000.41,2010-02-19 17:10:00 UTC,40.746105,-73.993745,40.738977,-73.989807,1
3130.3,2015-03-09 15:28:56 UTC,40.66441345214844,-74.00541687011719,40.66441345214844,-74.00541687011719,1
2023.7,2010-03-21 14:43:02 UTC,40.647064,-73.790056,40.65694,-73.917979,1
2010.9,2010-03-21 10:03:32 UTC,40.774846,-73.986265,40.743949,-73.971539,1
2004.5,2010-03-20 12:24:52 UTC,40.777854,-73.963134,40.782148,-73.975432,1
1564.5,2010-02-20 17:38:00 UTC,40.758255,-73.937783,40.758232,-73.937748,1
978.0,2015-01-12 01:41:43 UTC,40.68571090698242,-73.59175872802734,40.68571090698242,-73.59175872802734,4


In [0]:
from pyspark.sql import functions as F

final_df = saved_df.filter(
    F.col("fare_amount") <= 500
)

print("Final records:", final_df.count())
print("Removed extreme fares:", saved_df.count() - final_df.count())

Final records: 54033700
Removed extreme fares: 21


In [0]:
final_df.write \
    .mode("overwrite") \
    .parquet(
        "/Volumes/workspace/default/farewise_data/processed_final/"
    )

print("Final cleaned dataset saved!")

Final cleaned dataset saved!


In [0]:
final_df = spark.read.parquet(
    "/Volumes/workspace/default/farewise_data/processed_final/"
)

print("Total rows:", final_df.count())
print("Total columns:", len(final_df.columns))

final_df.printSchema()

Total rows: 54033700
Total columns: 15
root
 |-- weather_hour: timestamp (nullable = true)
 |-- key: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- temperature_2m (°C): double (nullable = true)
 |-- precipitation (mm): double (nullable = true)
 |-- rain (mm): double (nullable = true)
 |-- snowfall (cm): double (nullable = true)
 |-- wind_speed_10m (km/h): double (nullable = true)



In [0]:
final_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in [
        "fare_amount",
        "passenger_count",
        "pickup_ts",
        "temperature_2m (°C)",
        "precipitation (mm)"
    ]
]).show()

+-----------+---------------+---------+-------------------+------------------+
|fare_amount|passenger_count|pickup_ts|temperature_2m (°C)|precipitation (mm)|
+-----------+---------------+---------+-------------------+------------------+
|          0|              0|        0|                  0|                 0|
+-----------+---------------+---------+-------------------+------------------+



In [0]:
zones_df = (
    spark.read
    .option("multiLine", True)
    .json("/Volumes/workspace/default/farewise_data/zones/*.geojson")
)

zones_df.printSchema()

root
 |-- features: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- geometry: struct (nullable = true)
 |    |    |    |-- coordinates: array (nullable = true)
 |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- type: string (nullable = true)
 |    |    |-- properties: struct (nullable = true)
 |    |    |    |-- :created_at: string (nullable = true)
 |    |    |    |-- :id: string (nullable = true)
 |    |    |    |-- :updated_at: string (nullable = true)
 |    |    |    |-- :version: string (nullable = true)
 |    |    |    |-- borough: string (nullable = true)
 |    |    |    |-- locationid: string (nullable = true)
 |    |    |    |-- shape_area: string (nullable = true)
 |    |    |    |-- shape_leng

In [0]:
from pyspark.sql import functions as F

zones = zones_df.select(
    F.explode("features").alias("zone")
)

print("Total taxi zones:", zones.count())

display(
    zones.select(
        "zone.properties",
        "zone.geometry.type"
    ).limit(10)
)

Total taxi zones: 263


properties,type
"List(2025-06-09T17:12:22.062Z, row-rgxy_42jk_dq7g, 2025-06-09T17:12:22.062Z, rv-a7vs.vwuy~tyci, EWR, 1, 0.0007823067885, 0.116357453189, Newark Airport)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-76qp.92qd~k3iu, 2025-06-09T17:12:22.062Z, rv-2qzz~i35d~7f8b, Queens, 2, 0.00486634037837, 0.43346966679, Jamaica Bay)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-j55v~b782-bey7, 2025-06-09T17:12:22.062Z, rv-65ty-i8fr~zrr5, Bronx, 3, 0.000314414156821, 0.0843411059012, Allerton/Pelham Gardens)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-pnpa-ap69~6m7d, 2025-06-09T17:12:22.062Z, rv-wvih~xy8h~tpys, Manhattan, 4, 0.000111871946192, 0.0435665270921, Alphabet City)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-ewiw.9man~jjqn, 2025-06-09T17:12:22.062Z, rv-t95v.ubu4-6y4b, Staten Island, 5, 0.000497957489363, 0.0921464898574, Arden Heights)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-ux75~7ezj-9bkf, 2025-06-09T17:12:22.062Z, rv-q6yy_scgq_n4pn, Staten Island, 6, 0.000606460984581, 0.150490542523, Arrochar/Fort Wadsworth)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-24j9_vhb7~sp32, 2025-06-09T17:12:22.062Z, rv-9tnf-dcjc_6qex, Queens, 7, 0.000389787989274, 0.107417171123, Astoria)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-ssjn.m4tx_zaqw, 2025-06-09T17:12:22.062Z, rv-2dnv~y5bg~t7p6, Queens, 8, 0.000026587716279, 0.0275906911574, Astoria Park)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-8f58_znyg.8geg, 2025-06-09T17:12:22.062Z, rv-kk5c-nepg~4sep, Queens, 9, 0.000338443803197, 0.0997840924705, Auburndale)",MultiPolygon
"List(2025-06-09T17:12:22.062Z, row-uwcn_mnds-32bm, 2025-06-09T17:12:22.062Z, rv-b9yx~zmtw_68vn, Queens, 10, 0.000435823818081, 0.0998394794152, Baisley Park)",MultiPolygon


In [0]:
zones.select(
    "zone.properties",
    "zone.geometry"
).printSchema()

root
 |-- properties: struct (nullable = true)
 |    |-- :created_at: string (nullable = true)
 |    |-- :id: string (nullable = true)
 |    |-- :updated_at: string (nullable = true)
 |    |-- :version: string (nullable = true)
 |    |-- borough: string (nullable = true)
 |    |-- locationid: string (nullable = true)
 |    |-- shape_area: string (nullable = true)
 |    |-- shape_leng: string (nullable = true)
 |    |-- zone: string (nullable = true)
 |-- geometry: struct (nullable = true)
 |    |-- coordinates: array (nullable = true)
 |    |    |-- element: array (containsNull = true)
 |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |-- element: array (containsNull = true)
 |    |    |    |    |    |-- element: double (containsNull = true)
 |    |-- type: string (nullable = true)

